In [ ]:
%pip show langchain
%pip show langchain-community
%pip show rank-bm25

In [ ]:
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_chroma import Chroma
from langchain_core.documents import Document

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Load PDF
loader = PyPDFLoader("docs/attention-is-all-you-need.pdf")
documents = loader.load()

print(f"Loaded {len(documents)} pages")

# Split into chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = splitter.split_documents(documents)

print(f"Created {len(chunks)} chunks")

In [ ]:
embedding_model = OllamaEmbeddings(
    model="nomic-embed-text"
)

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    collection_metadata={"hnsw:space": "cosine"}
)

vector_retriever = vectorstore.as_retriever(
    search_kwargs={"k": 2}
)

print("Vector retriever ready!")

In [ ]:
test_query = "What is the Transformer architecture?"

print(f"Testing: '{test_query}'")

test_docs = vector_retriever.invoke(test_query)

for doc in test_docs:
    print(f"Found: {doc.page_content}")

In [ ]:
# BM25 Retriever (Keyword Search / Sparse Retrieval)
print("Setting up BM25 Retriever...")

bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 3

print("BM25 retriever ready!")

In [ ]:
test_query = "multi-head attention"

print(f"Testing: '{test_query}'")

test_docs = bm25_retriever.invoke(test_query)

for doc in test_docs:
    print(f"Found: {doc.page_content}")

In [ ]:
# Hybrid retrieval test

test_query = "What is the Transformer architecture?"

retrieved_chunks = hybrid_retriever.invoke(test_query)

print("Retrieved Documents:\n")

for i, doc in enumerate(retrieved_chunks, 1):
    print(f"{i}. {doc.page_content}")

In [ ]:
# Vector search
test_query = "What is the Transformer architecture?"

test_docs = vector_retriever.invoke(test_query)

for doc in test_docs:
    print(doc.page_content)

In [ ]:
# BM25 search
test_query = "multi-head attention"

test_docs = bm25_retriever.invoke(test_query)

for doc in test_docs:
    print(doc.page_content)

In [ ]:
# Hybrid search
test_query = "What is the purpose of multi-head attention?"

retrieved_chunks = hybrid_retriever.invoke(test_query)

for i, doc in enumerate(retrieved_chunks, 1):
    print(f"\n--- Chunk {i} ---")
    print(doc.page_content)

In [ ]:
from langchain_ollama import ChatOllama

model = ChatOllama(
    model="phi3",
    temperature=0
)

print("Phi-3 ready!")

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage

# Combine the query and retrieved documents

combined_input = f"""
Based on the following documents, please answer this question:

Question:
{test_query}

Documents:
{chr(10).join([f"- {doc.page_content}" for doc in retrieved_chunks])}

Instructions:
- Answer using only the information from the provided documents.
- Do not make up information.
- If the documents do not contain enough information, say:
  "I don't have enough information to answer that question based on the provided documents."
- Give a clear and concise answer.
"""

messages = [
    SystemMessage(
        content="You are a helpful assistant that answers questions using provided documents."
    ),
    HumanMessage(
        content=combined_input
    )
]

result = model.invoke(messages)

print("\n--- Generated Response ---")
print(result.content)